In [1]:
!pip install langchain langchain-community langchain-google-genai langchain-huggingface streamlit pymupdf sentence-transformers python-dotenv chromadb


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import langchain 
import chromadb
import fitz
import streamlit
import os
from pathlib import Path
print("langchain",langchain.__version__)
print("Chromadb",chromadb.__version__)

langchain 1.2.15
Chromadb 1.5.5


In [3]:
def load_pdfs(data_folder="data"):
    documents = []  # will hold all extracted text + metadata
    
    pdf_files = list(Path(data_folder).glob("*.pdf"))
    
    if not pdf_files:
        print("No PDFs found in /data folder!")
        return []
    
    for pdf_path in pdf_files:
        print(f"Reading: {pdf_path.name}")
        
        doc = fitz.open(pdf_path)  # open the PDF
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            text = page.get_text().strip()  # extract text
            
            if not text:  # skip blank pages
                continue
            
            # store text + metadata together
            documents.append({
                "text": text,
                "metadata": {
                    "source": pdf_path.name,
                    "page": page_num + 1
                }
            })
        print(f"Done — {len(doc)} pages extracted from {pdf_path.name} ")
        doc.close()
            
    
    return documents

In [4]:
documents=load_pdfs("data")
print(f"\nTotal pages extracted: {len(documents)}")
print(f"\nPreview of Page 1")
print(f"Source: {documents[0]['metadata']['source']}")
print(f"Page:{documents[0]['metadata']['page']}")
print(f"Text : {documents[0]['text'][:300]}.....")

Reading: NIPS-2017-attention-is-all-you-need-Paper.pdf
Done — 11 pages extracted from NIPS-2017-attention-is-all-you-need-Paper.pdf 

Total pages extracted: 11

Preview of Page 1
Source: NIPS-2017-attention-is-all-you-need-Paper.pdf
Page:1
Text : Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aid.....


In [5]:
!pip install -U langchain-text-splitters


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("Splitter Imported")

Splitter Imported


In [7]:
def chunk_documents(documents,chunk_size=1000,chunk_overlap=50):
    splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n","\n","."," "])
    all_chunks = []
    for doc in documents:
        chunks=splitter.split_text(doc["text"])
        for chunk in chunks:
            all_chunks.append({
                "text":chunk,
                "metadata":doc["metadata"]
            })
    return all_chunks


In [8]:
chunks = chunk_documents(documents)

print(f"Total pages   : {len(documents)}")
print(f"Total chunks  : {len(chunks)}")
print(f"Avg per page  : {len(chunks) // len(documents)} chunks/page")
print(f"\n--- Preview of Chunk 1 ---")
print(f"Source : {chunks[0]['metadata']['source']}")
print(f"Page   : {chunks[0]['metadata']['page']}")
print(f"Text   : {chunks[0]['text']}")
print(f"\n--- Preview of Chunk 2 ---")
print(f"Text   : {chunks[1]['text']}")

Total pages   : 11
Total chunks  : 39
Avg per page  : 3 chunks/page

--- Preview of Chunk 1 ---
Source : NIPS-2017-attention-is-all-you-need-Paper.pdf
Page   : 1
Text   : Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experime

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [10]:
# grab just the text from first chunk
sample_text = chunks[0]['text']

# embed it
vector = embedding_model.embed_query(sample_text)

print(f"Sample text : {sample_text[:100]}...")
print(f"Vector size : {len(vector)} dimensions")
print(f"First 5 nums: {vector[:5]}")

Sample text : Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brai...
Vector size : 384 dimensions
First 5 nums: [-0.07986008375883102, -0.12790851294994354, 0.015592114999890327, -0.032777052372694016, 0.018648797646164894]


In [11]:
pip install -qU "langchain-chroma>=0.1.2"

Note: you may need to restart the kernel to use updated packages.


In [12]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
print("ChromaDB imported")

ChromaDB imported


In [13]:
def convert_to_langchain_docs(chunks):
    langchain_docs = []
    
    for chunk in chunks:
        doc = Document(
            page_content=chunk["text"],
            metadata=chunk["metadata"]
        )
        langchain_docs.append(doc)
    
    return langchain_docs

langchain_docs = convert_to_langchain_docs(chunks)

print(f"Total LangChain docs : {len(langchain_docs)}")
print(f"\n--- Preview ---")
print(f"Content  : {langchain_docs[0].page_content[:150]}...")
print(f"Metadata : {langchain_docs[0].metadata}")

Total LangChain docs : 39

--- Preview ---
Content  : Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nik...
Metadata : {'source': 'NIPS-2017-attention-is-all-you-need-Paper.pdf', 'page': 1}


In [14]:
vector_store=Chroma.from_documents(
    documents=langchain_docs,
    embedding=embedding_model,
    persist_directory="chroma_db"
)
print("All chunks embedded in ChromaDB")

All chunks embedded in ChromaDB


In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()

True

In [16]:
llm=ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key= os.getenv("GOOGLE_API_KEY"),
    temperature=0.3
)
print("Gemini flash loaded")

Gemini flash loaded


In [17]:
prompt=ChatPromptTemplate.from_template(""" You are an expert research assistant helping users find information from academic research papers.
Use ONLY the following context to answer the question.
If the answer is not in the context, say "I could not find this information in the uploaded papers".
Always mention which paper and page number your answer comes from.
Context: {context}
Question: {question}
Answer:
"""
)
print("prompt created")

prompt created


In [18]:
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)
print("Retriever created")

Retriever created


In [19]:
def format_docs(docs):
    formatted = []
    for doc in docs:
        formatted.append(
            f"Source: {doc.metadata['source']} | "
            f"Page: {doc.metadata['page']}\n"
            f"{doc.page_content}"
        )
    return "\n\n---\n\n".join(formatted)

In [20]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def get_context(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

def build_prompt_input(question):
    return {
        "context": get_context(question),
        "question": question
    }

rag_chain = (
    RunnableLambda(build_prompt_input)
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built ✅")

RAG chain built ✅


In [21]:
def ask(question):
    print(f"Question: {question}")
    print(f"\nThinking...\n")
    
    answer = rag_chain.invoke(question)
    
    print(f"Answer:\n{answer}")
    print("\n" + "="*60 + "\n")

# Try it!
ask("What datasets were used for training?")


Question: What datasets were used for training?

Thinking...

Answer:
Based on the provided context, the following datasets were used for training:

*   **WMT 2014 English-German dataset:** This standard dataset consists of about 4.5 million sentence pairs.
*   **WMT 2014 English-French dataset:** This significantly larger dataset consists of 36 million sentences.

(Source: NIPS-2017-attention-is-all-you-need-Paper.pdf | Page: 7)


